# Tutorial 1: Environments and Neural Networks

**zrth** extracts symbolic reactive modules from plain Python classes. You write a standard [Gymnasium](https://gymnasium.farama.org/) environment or a [PyTorch](https://pytorch.org/) neural network — zrth analyzes it and produces a formal model with typed wires and terms that can be used for verification or training.

This tutorial walks through:
1. Defining an environment (`Env`)
2. Defining a neural network (`NN`)
3. Inspecting the extracted modules

## Defining an environment

To define an environment, subclass `Env` (which extends `gym.Env`):

- **`__init__`**: set `action_space`, `observation_space`, any parameters (e.g. `self.y0`), and state variables (e.g. `self.x`)
- **`reset`**: initialize all state variables, return `(observation, reward, terminated, truncated)`
- **`step`**: update state given an action, return the same four-tuple

**Important:** state variables must also be assigned in `__init__` so the analyzer can infer their types. For example, even though `reset` sets `self.x = 0.0`, you should also write `self.x = 0.0` in `__init__`.

Below is a counter system with three variables $x$, $y$, $z$. The variable $x$ increments while $x < y \lor x < z$, and resets to $0$ otherwise. We want to show that $x = y \lor x = z$ must occur infinitely often.

In [ ]:
import torch
import torch.nn as nn
from gymnasium import spaces
from zrth import Env, NN


class CounterEnv(Env):
    """Counter environment: x increments while x < y or x < z, resets otherwise."""

    def __init__(self, y0=5.0, z0=3.0):
        super().__init__()

        self.action_space = spaces.Discrete(1)
        self.observation_space = spaces.Box(low=-1e6, high=1e6, shape=(1,))

        self.y0 = y0
        self.z0 = z0

        # Also set state variables here so the analyzer can infer their dtype
        self.x = 0.0
        self.y = y0
        self.z = z0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.x = 0.0
        self.y = self.y0
        self.z = self.z0

        observation = self.x
        reward = 0.0
        terminated = False
        truncated = False
        return observation, reward, terminated, truncated

    def step(self, action):
        if self.x < self.y or self.x < self.z:
            self.x = self.x + 1.0
        else:
            self.x = 0.0

        # y and z remain constant
        self.y = self.y
        self.z = self.z

        at_target = self.x == self.y or self.x == self.z
        reward = 1.0 if at_target else 0.0
        terminated = at_target
        truncated = False
        observation = self.x
        return observation, reward, terminated, truncated

## Instantiation

When you call `CounterEnv(...)`, zrth analyzes `__init__`, `reset`, and `step` behind the scenes. It:
1. Infers action/observation types from the gymnasium spaces
2. Classifies `self.*` attributes as **private** (read+write state) or **parameters** (read-only constants)
3. Creates typed wire pairs for each variable and signal
4. Converts `reset` into **init** terms and `step` into **update** terms

Print the module to see the result.

In [ ]:
counter = CounterEnv(y0=5.0, z0=3.0)
print(counter)

## Reading the output

The printed module shows:

- **external**: input wires — here, the action (unused by this environment, but still present)
- **interface**: output wires — observation, reward, terminated, truncated
- **private**: internal state — wire pairs `wN, wM` for x, y, z (each pair is `latched, next`)
- **init**: terms from `reset` — how each wire is initialized
- **update**: terms from `step` — how each wire is updated, including `Ite` (if-then-else), `Lt`, `Eq`, `Add`

## Defining a neural network

To define a neural network, subclass `NN` (which extends `nn.Module`):

- **`__init__`**: define `nn.Linear` layers
- **`forward`**: standard PyTorch forward pass

zrth extracts the architecture as a **combinatorial** (stateless) module. The resulting module has:
- **extl** (external input) wire pair — sized from the first layer's `in_features`
- **intf** (interface output) wire pair — sized from the last layer's `out_features`

The ranking function below maps $(x, y, z) \in \mathbb{R}^3$ to a scalar. It's a verification artifact — trained to decrease on each step until the liveness property $x = y \lor x = z$ holds — not a controller, so we don't compose it with the counter.

In [ ]:
class RankingNN(NN):
    """Ranking function: R(x, y, z) -> scalar.

    Architecture: [3] -> 2 -> [1]
    """

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 2)
        self.fc2 = nn.Linear(2, 1)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        return self.fc2(x)

Instantiate and inspect. Note that the NN module has no `private` wires (it's stateless) and the `init`/`update` blocks are identical (combinatorial — same computation every step).

In [ ]:
from zrth import Module

ranking = RankingNN()
print(Module.__str__(ranking))

## Training the ranking function

A **ranking function** $R: \mathcal{S} \to \mathbb{R}$ proves that a liveness property holds infinitely often. The key conditions are:

1. $R(s) \geq 0$ for all states $s$
2. $R(s) = 0$ when the liveness property holds (here: $x = y \lor x = z$)
3. $R(s') < R(s)$ whenever the property does *not* hold (the ranking strictly decreases)

If such an $R$ exists, the property must hold infinitely often — otherwise $R$ would decrease forever below zero, contradicting condition 1.

We train the `RankingNN` to satisfy these conditions by:
1. Simulating the counter for $N$ steps (it's a fully functional `gym.Env`)
2. At each step, computing $R(x, y, z)$ and $R(x', y', z')$
3. Penalizing violations: $R$ should decrease at non-target states and be near zero at target states

In [ ]:
ranking = RankingNN()
optimizer = torch.optim.Adam(ranking.parameters(), lr=0.01)
margin = 0.1
n_epochs = 200
n_steps = 20

for epoch in range(n_epochs):
    # Simulate the counter to collect a trajectory of (x, y, z) states.
    # CounterEnv is a full gym.Env — reset() and step() work directly.
    counter = CounterEnv(y0=5.0, z0=3.0)
    counter.reset()

    states = []
    for _ in range(n_steps):
        states.append((counter.x, counter.y, counter.z))
        counter.step(0)  # Discrete(1) action space — always 0

    # Compute ranking loss over consecutive state pairs
    loss = torch.tensor(0.0)
    for i in range(len(states) - 1):
        s = torch.tensor(states[i], dtype=torch.float32)
        s_next = torch.tensor(states[i + 1], dtype=torch.float32)
        r = ranking(s.unsqueeze(0)).squeeze()
        r_next = ranking(s_next.unsqueeze(0)).squeeze()

        x, y, z = states[i]
        at_target = (x == y) or (x == z)

        if not at_target:
            # R must strictly decrease by at least `margin`
            loss = loss + torch.relu(r_next - r + margin)
            # R must be non-negative
            loss = loss + torch.relu(-r)
        else:
            # R should be ~0 at target states
            loss = loss + r ** 2

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"epoch {epoch:3d}  loss = {loss.item():.4f}")

## Verifying the ranking function

After training, we evaluate $R$ along a trajectory. At non-target states, $R$ should decrease on each step. At target states ($x = y$ or $x = z$), $R$ should be near zero.

In [ ]:
counter = CounterEnv(y0=5.0, z0=3.0)
counter.reset()

print(f"{'step':>4}  {'x':>5} {'y':>5} {'z':>5}  {'R(s)':>8}  {'target':>6}")
print("-" * 45)

with torch.no_grad():
    for step in range(12):
        x, y, z = counter.x, counter.y, counter.z
        s = torch.tensor([x, y, z], dtype=torch.float32)
        r = ranking(s.unsqueeze(0)).squeeze().item()
        at_target = (x == y) or (x == z)
        print(f"{step:4d}  {x:5.1f} {y:5.1f} {z:5.1f}  {r:8.4f}  {'  *' if at_target else ''}")
        counter.step(0)